# Week 15: Agent Studio I: build a bounded workflow - Exercise

Machine Learning in Engineering Science | NCKU + LAiMM | Fall 2026

**Meeting:** Monday 2026-12-14, 14:00-17:00. Three one-hour modules; 60 minutes explanation and 120 minutes practice across slides and this notebook.

**Learning outcome:** Build a source-bounded thermal-report workflow from synthetic readings with explicit units, checks and review status.

**Scope:** Implement a deterministic engineering-report baseline using original synthetic temperature readings and a fictional review protocol. No physical diagnosis, equipment command or live LLM inference is performed.

**Free path:** use Python 3, a CPU Colab runtime or Jupyter. No API key, GPU, paid inference, confidential data or external dataset download is required. First-time package installation needs internet. Record actual versions; do not assume a Colab version.

**How to work:** read a task; edit its prediction field before running; run its code with Shift+Enter; inspect the output; write an evidence-based explanation. Core scaffolds run as supplied so you can start without filling a programming puzzle. Make only the requested controlled change. Use an AI coding assistant only if desired and disclose what you checked.

**Practice policy:** 8 core tasks and 2 optional challenges. In-class practice, not additional weekly graded homework. Weeks 11 and 18 remain written and oral examinations.

**Recovery:** if a variable is missing, restart and run all from the top. Initialization resets responses; after editing answers, rerun only the final export cell. Save your own edited notebook as well as the ZIP; the exported analysis source is the supplied scaffold snapshot, not a capture of unsaved code edits.

## 0. Environment and response record

Run this once at the beginning. It installs only missing free packages, creates a week-specific artifact directory, and initializes 19 blank reflection fields. Printouts are observations, not answer keys.

In [ ]:
WEEK=15
WORKFLOW_TEXT="\nimport math, statistics, hashlib, json\nVERSION='thermal-workflow-1.0'\nPROTOCOL={'id':'LAB-P1:v1','threshold_C':80.0,'comparison':'>=','scope':'Fictional original teaching protocol; no hardware authority'}\ndef stable_hash(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False,separators=(',',':')).encode()).hexdigest()\ndef analyze(request,protocol=PROTOCOL,bug=False):\n    trace=[]\n    def finish(status,reason=None,**kw):\n        return dict(status=status,reason=reason,trace=trace,**kw)\n    if type(request) is not dict or set(request)!={'case_id','unit','values'}:\n        return finish('rejected','exact request fields required')\n    if type(request['case_id']) is not str or not request['case_id'].strip():\n        return finish('rejected','nonblank string case ID required')\n    values=request['values']\n    if type(request['unit']) is not str or request['unit'] not in ('C','F'):\n        return finish('rejected','unit must be C or F')\n    if type(values) is not list or not 2<=len(values)<=100 or not all(type(v) in (int,float) and abs(v)<=1e6 and math.isfinite(v) for v in values):\n        return finish('rejected','2..100 finite bounded numeric readings required')\n    trace.append({'stage':'validate','status':'ok','input_sha256':stable_hash(request)})\n    c=[float(v) if request['unit']=='C' else (float(v)-32)*5/9 for v in values]\n    trace.append({'stage':'convert','unit':'C','n':len(c)})\n    mean=(sum(c)/(len(c)+1)) if bug else statistics.mean(c)\n    sd=statistics.stdev(c)\n    trace.append({'stage':'summarize','mean_C':mean,'sample_sd_C':sd})\n    if (type(protocol) is not dict or set(protocol)!={'id','threshold_C','comparison','scope'} or\n        type(protocol.get('id')) is not str or not protocol['id'].strip() or\n        type(protocol.get('threshold_C')) not in (int,float) or abs(protocol['threshold_C'])>1e6 or not math.isfinite(protocol['threshold_C']) or\n        protocol.get('comparison')!='>=' or type(protocol.get('scope')) is not str):\n        return finish('abstain','approved class protocol unavailable',case_id=request['case_id'],mean_C=mean,sample_sd_C=sd)\n    trace.append({'stage':'protocol','source_id':protocol['id'],'threshold_C':protocol['threshold_C']})\n    decision_value=round(mean,10)\n    status='review_required' if decision_value>=protocol['threshold_C'] else 'reported'\n    trace.append({'stage':'report','status':status})\n    return finish(status,case_id=request['case_id'],mean_C=mean,sample_sd_C=sd,decision_value_C=decision_value,\n                  input_sha256=stable_hash(request),protocol_id=protocol['id'],version=VERSION,\n                  limitation='Synthetic class readings and fictional protocol; not a physical safety conclusion.')\n"
RUNNER_TEXT=''
ANALYSIS_SOURCE='WEEK=15\nWORKFLOW_TEXT="\\nimport math, statistics, hashlib, json\\nVERSION=\'thermal-workflow-1.0\'\\nPROTOCOL={\'id\':\'LAB-P1:v1\',\'threshold_C\':80.0,\'comparison\':\'>=\',\'scope\':\'Fictional original teaching protocol; no hardware authority\'}\\ndef stable_hash(obj):\\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False,separators=(\',\',\':\')).encode()).hexdigest()\\ndef analyze(request,protocol=PROTOCOL,bug=False):\\n    trace=[]\\n    def finish(status,reason=None,**kw):\\n        return dict(status=status,reason=reason,trace=trace,**kw)\\n    if type(request) is not dict or set(request)!={\'case_id\',\'unit\',\'values\'}:\\n        return finish(\'rejected\',\'exact request fields required\')\\n    if type(request[\'case_id\']) is not str or not request[\'case_id\'].strip():\\n        return finish(\'rejected\',\'nonblank string case ID required\')\\n    values=request[\'values\']\\n    if type(request[\'unit\']) is not str or request[\'unit\'] not in (\'C\',\'F\'):\\n        return finish(\'rejected\',\'unit must be C or F\')\\n    if type(values) is not list or not 2<=len(values)<=100 or not all(type(v) in (int,float) and abs(v)<=1e6 and math.isfinite(v) for v in values):\\n        return finish(\'rejected\',\'2..100 finite bounded numeric readings required\')\\n    trace.append({\'stage\':\'validate\',\'status\':\'ok\',\'input_sha256\':stable_hash(request)})\\n    c=[float(v) if request[\'unit\']==\'C\' else (float(v)-32)*5/9 for v in values]\\n    trace.append({\'stage\':\'convert\',\'unit\':\'C\',\'n\':len(c)})\\n    mean=(sum(c)/(len(c)+1)) if bug else statistics.mean(c)\\n    sd=statistics.stdev(c)\\n    trace.append({\'stage\':\'summarize\',\'mean_C\':mean,\'sample_sd_C\':sd})\\n    if (type(protocol) is not dict or set(protocol)!={\'id\',\'threshold_C\',\'comparison\',\'scope\'} or\\n        type(protocol.get(\'id\')) is not str or not protocol[\'id\'].strip() or\\n        type(protocol.get(\'threshold_C\')) not in (int,float) or abs(protocol[\'threshold_C\'])>1e6 or not math.isfinite(protocol[\'threshold_C\']) or\\n        protocol.get(\'comparison\')!=\'>=\' or type(protocol.get(\'scope\')) is not str):\\n        return finish(\'abstain\',\'approved class protocol unavailable\',case_id=request[\'case_id\'],mean_C=mean,sample_sd_C=sd)\\n    trace.append({\'stage\':\'protocol\',\'source_id\':protocol[\'id\'],\'threshold_C\':protocol[\'threshold_C\']})\\n    decision_value=round(mean,10)\\n    status=\'review_required\' if decision_value>=protocol[\'threshold_C\'] else \'reported\'\\n    trace.append({\'stage\':\'report\',\'status\':status})\\n    return finish(status,case_id=request[\'case_id\'],mean_C=mean,sample_sd_C=sd,decision_value_C=decision_value,\\n                  input_sha256=stable_hash(request),protocol_id=protocol[\'id\'],version=VERSION,\\n                  limitation=\'Synthetic class readings and fictional protocol; not a physical safety conclusion.\')\\n"\nRUNNER_TEXT=\'\'\n\nimport sys, subprocess, importlib.util, platform, json, math, hashlib, statistics\nimport tempfile, zipfile, shutil, copy, random\nfrom pathlib import Path\nmissing=[p for m,p in [(\'numpy\',\'numpy\'),(\'matplotlib\',\'matplotlib\'),(\'sklearn\',\'scikit-learn\')] if importlib.util.find_spec(m) is None]\nif missing:\n    subprocess.check_call([sys.executable,\'-m\',\'pip\',\'install\',*missing])\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\nimport sklearn\nSEED=2026\nrng=np.random.default_rng(SEED)\nrandom.seed(SEED)\nARTIFACTS=Path(\'artifacts\')/f\'week{WEEK}\'\nARTIFACTS.mkdir(parents=True,exist_ok=True)\nENV={\'python\':platform.python_version(),\'numpy\':np.__version__,\'matplotlib\':matplotlib.__version__,\'scikit_learn\':sklearn.__version__,\'platform\':platform.platform()}\ndef canonical(x): return json.dumps(x,sort_keys=True,allow_nan=False,separators=(\',\',\':\')).encode()\ndef digest(x): return hashlib.sha256(canonical(x)).hexdigest()\ndef save_json(name,obj):\n    p=ARTIFACTS/name; p.write_text(json.dumps(obj,indent=2,allow_nan=False)); return p\ndef plot(name,labels,values,ylabel,title):\n    fig,ax=plt.subplots(figsize=(9,4.8),layout=\'constrained\')\n    ax.bar(labels,values,color=[\'#167b87\',\'#7f2635\',\'#b88a29\',\'#12233f\']*10)\n    ax.set_ylabel(ylabel); ax.set_title(title)\n    ax.tick_params(axis=\'x\',labelrotation=18); ax.spines[[\'top\',\'right\']].set_visible(False)\n    fig.savefig(ARTIFACTS/name,dpi=160); plt.show(); plt.close(fig)\nobservations={}\nresponses={**{f\'task{i}_{p}\':\'\' for i in range(1,9) for p in [\'prediction\',\'explanation\']},\'controlled_change\':\'\',\'limitation\':\'\',\'ai_disclosure\':\'\'}\nprint(\'Environment:\',ENV,\'; seed\',SEED)\n\n\nimport math, statistics, hashlib, json\nVERSION=\'thermal-workflow-1.0\'\nPROTOCOL={\'id\':\'LAB-P1:v1\',\'threshold_C\':80.0,\'comparison\':\'>=\',\'scope\':\'Fictional original teaching protocol; no hardware authority\'}\ndef stable_hash(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False,separators=(\',\',\':\')).encode()).hexdigest()\ndef analyze(request,protocol=PROTOCOL,bug=False):\n    trace=[]\n    def finish(status,reason=None,**kw):\n        return dict(status=status,reason=reason,trace=trace,**kw)\n    if type(request) is not dict or set(request)!={\'case_id\',\'unit\',\'values\'}:\n        return finish(\'rejected\',\'exact request fields required\')\n    if type(request[\'case_id\']) is not str or not request[\'case_id\'].strip():\n        return finish(\'rejected\',\'nonblank string case ID required\')\n    values=request[\'values\']\n    if type(request[\'unit\']) is not str or request[\'unit\'] not in (\'C\',\'F\'):\n        return finish(\'rejected\',\'unit must be C or F\')\n    if type(values) is not list or not 2<=len(values)<=100 or not all(type(v) in (int,float) and abs(v)<=1e6 and math.isfinite(v) for v in values):\n        return finish(\'rejected\',\'2..100 finite bounded numeric readings required\')\n    trace.append({\'stage\':\'validate\',\'status\':\'ok\',\'input_sha256\':stable_hash(request)})\n    c=[float(v) if request[\'unit\']==\'C\' else (float(v)-32)*5/9 for v in values]\n    trace.append({\'stage\':\'convert\',\'unit\':\'C\',\'n\':len(c)})\n    mean=(sum(c)/(len(c)+1)) if bug else statistics.mean(c)\n    sd=statistics.stdev(c)\n    trace.append({\'stage\':\'summarize\',\'mean_C\':mean,\'sample_sd_C\':sd})\n    if (type(protocol) is not dict or set(protocol)!={\'id\',\'threshold_C\',\'comparison\',\'scope\'} or\n        type(protocol.get(\'id\')) is not str or not protocol[\'id\'].strip() or\n        type(protocol.get(\'threshold_C\')) not in (int,float) or abs(protocol[\'threshold_C\'])>1e6 or not math.isfinite(protocol[\'threshold_C\']) or\n        protocol.get(\'comparison\')!=\'>=\' or type(protocol.get(\'scope\')) is not str):\n        return finish(\'abstain\',\'approved class protocol unavailable\',case_id=request[\'case_id\'],mean_C=mean,sample_sd_C=sd)\n    trace.append({\'stage\':\'protocol\',\'source_id\':protocol[\'id\'],\'threshold_C\':protocol[\'threshold_C\']})\n    decision_value=round(mean,10)\n    status=\'review_required\' if decision_value>=protocol[\'threshold_C\'] else \'reported\'\n    trace.append({\'stage\':\'report\',\'status\':status})\n    return finish(status,case_id=request[\'case_id\'],mean_C=mean,sample_sd_C=sd,decision_value_C=decision_value,\n                  input_sha256=stable_hash(request),protocol_id=protocol[\'id\'],version=VERSION,\n                  limitation=\'Synthetic class readings and fictional protocol; not a physical safety conclusion.\')\n\nREQUEST={\'case_id\':\'synthetic-A\',\'unit\':\'C\',\'values\':[78.,80.,82.]}\nprint(\'Protocol:\',PROTOCOL)\n\nrequests=[{\'case_id\':\'low\',\'unit\':\'C\',\'values\':[70,72,74]}, {\'case_id\':\'equal\',\'unit\':\'C\',\'values\':[78,80,82]},\n          {\'case_id\':\'high\',\'unit\':\'C\',\'values\':[81,82,83]}, {\'case_id\':\'bad\',\'values\':[1,2]}]\nresults=[analyze(r) for r in requests]\nassert [r[\'status\'] for r in results]==[\'reported\',\'review_required\',\'review_required\',\'rejected\']\nobservations[\'task1\']={\'requests\':requests,\'results\':results}\nplot(\'case-means.png\',[\'low\',\'equal\',\'high\'],[r[\'mean_C\'] for r in results[:3]],\'Mean (C)\',\'Synthetic repeated-reading cases; fictional threshold 80 C\')\nprint([r[\'status\'] for r in results])\nresult=analyze(REQUEST)\nobservations[\'task2\']={k:result[k] for k in [\'case_id\',\'input_sha256\',\'protocol_id\',\'version\',\'limitation\']}\nassert result[\'input_sha256\']==stable_hash(REQUEST) and result[\'protocol_id\']==\'LAB-P1:v1\'\nprint(observations[\'task2\'])\nf={\'case_id\':\'synthetic-A-F\',\'unit\':\'F\',\'values\':[172.4,176.,179.6]}\nc_result=analyze(REQUEST); f_result=analyze(f)\nassert math.isclose(c_result[\'mean_C\'],f_result[\'mean_C\'],abs_tol=1e-10)\nassert math.isclose(c_result[\'sample_sd_C\'],f_result[\'sample_sd_C\'],abs_tol=1e-10)\nassert c_result[\'status\']==f_result[\'status\']==\'review_required\'\nobservations[\'task3\']={\'C\':c_result,\'F\':f_result,\'mean_difference\':f_result[\'mean_C\']-c_result[\'mean_C\']}\nprint(observations[\'task3\'])\na=np.array(REQUEST[\'values\']); deviations=a-a.mean(); squares=deviations**2\nmanual=math.sqrt(float(squares.sum())/(len(a)-1))\nassert manual==2 and math.isclose(analyze(REQUEST)[\'sample_sd_C\'],manual)\nobservations[\'task4\']={\'deviations\':deviations.tolist(),\'squares\':squares.tolist(),\'denominator\':len(a)-1,\'sample_sd_C\':manual}\nplot(\'deviations.png\',[\'r1\',\'r2\',\'r3\'],deviations,\'Deviation from mean (C)\',\'Synthetic repeated readings, not independent deployment cases\')\nprint(observations[\'task4\'])\nbad=copy.deepcopy(REQUEST); bad[\'unit\']=\'unknown\'\nfailed=analyze(bad)\nassert failed[\'status\']==\'rejected\' and failed[\'trace\']==[] and \'mean_C\' not in failed\nobservations[\'task5\']={\'request\':bad,\'result\':failed,\'downstream_stages_present\':[]}; print(observations[\'task5\'])\nmissing=analyze(REQUEST,protocol=None)\nassert missing[\'status\']==\'abstain\' and missing[\'mean_C\']==80 and \'protocol_id\' not in missing\nassert [t[\'stage\'] for t in missing[\'trace\']]==[\'validate\',\'convert\',\'summarize\']\nobservations[\'task6\']={\'result\':missing}; print(missing)\nfixture={\'case_id\':\'boundary\',\'unit\':\'C\',\'values\':[79,80,81]}\nreference=analyze(fixture); buggy=analyze(fixture,bug=True)\nassert reference[\'mean_C\']==80 and buggy[\'mean_C\']==60\nassert reference[\'status\']==\'review_required\' and buggy[\'status\']==\'reported\'\nobservations[\'task7\']={\'fixture\':fixture,\'reference\':reference,\'buggy\':buggy}; print(observations[\'task7\'])\nsave_json(\'request.json\',REQUEST); save_json(\'protocol.json\',PROTOCOL)\nobservations[\'task8\']={\'request_sha256\':stable_hash(REQUEST),\'protocol_sha256\':stable_hash(PROTOCOL),\n \'trace\':analyze(REQUEST)[\'trace\'],\'scope\':\'Local deterministic baseline; original synthetic readings; fictional protocol.\'}\nassert set(observations)=={f\'task{i}\' for i in range(1,9)}\nprint(observations[\'task8\'])\nRUN_CHALLENGES=False\n\noptional={}\nif RUN_CHALLENGES:\n    # Explicit adapter: Kelvin is not silently accepted by the original C/F contract.\n    kelvin={\'case_id\':\'K-adapted\',\'unit\':\'K\',\'values\':[351.15,353.15,355.15]}\n    adapted={\'case_id\':kelvin[\'case_id\'],\'unit\':\'C\',\'values\':[v-273.15 for v in kelvin[\'values\']]}\n    assert analyze(kelvin)[\'status\']==\'rejected\'\n    kr=analyze(adapted); assert math.isclose(kr[\'mean_C\'],80,abs_tol=1e-10) and kr[\'status\']==\'review_required\'\n    optional[\'A_kelvin_adapter\']={\'original\':kelvin,\'adapted\':adapted,\'result\':kr}\n    reversed_request=copy.deepcopy(REQUEST); reversed_request[\'values\'].reverse()\n    rr=analyze(reversed_request); original=analyze(REQUEST)\n    assert math.isclose(rr[\'mean_C\'],original[\'mean_C\'],abs_tol=1e-10) and math.isclose(rr[\'sample_sd_C\'],original[\'sample_sd_C\'],abs_tol=1e-10)\n    optional[\'B_order_invariance\']={\'result\':rr,\'note\':\'Numerical summary invariant; raw input digest and order differ.\'}\nprint(optional)\n'

import sys, subprocess, importlib.util, platform, json, math, hashlib, statistics
import tempfile, zipfile, shutil, copy, random
from pathlib import Path
missing=[p for m,p in [('numpy','numpy'),('matplotlib','matplotlib'),('sklearn','scikit-learn')] if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
SEED=2026
rng=np.random.default_rng(SEED)
random.seed(SEED)
ARTIFACTS=Path('artifacts')/f'week{WEEK}'
ARTIFACTS.mkdir(parents=True,exist_ok=True)
ENV={'python':platform.python_version(),'numpy':np.__version__,'matplotlib':matplotlib.__version__,'scikit_learn':sklearn.__version__,'platform':platform.platform()}
def canonical(x): return json.dumps(x,sort_keys=True,allow_nan=False,separators=(',',':')).encode()
def digest(x): return hashlib.sha256(canonical(x)).hexdigest()
def save_json(name,obj):
    p=ARTIFACTS/name; p.write_text(json.dumps(obj,indent=2,allow_nan=False)); return p
def plot(name,labels,values,ylabel,title):
    fig,ax=plt.subplots(figsize=(9,4.8),layout='constrained')
    ax.bar(labels,values,color=['#167b87','#7f2635','#b88a29','#12233f']*10)
    ax.set_ylabel(ylabel); ax.set_title(title)
    ax.tick_params(axis='x',labelrotation=18); ax.spines[['top','right']].set_visible(False)
    fig.savefig(ARTIFACTS/name,dpi=160); plt.show(); plt.close(fig)
observations={}
responses={**{f'task{i}_{p}':'' for i in range(1,9) for p in ['prediction','explanation']},'controlled_change':'','limitation':'','ai_disclosure':''}
print('Environment:',ENV,'; seed',SEED)

## 1. Inspect the reference mechanism

Implement a deterministic engineering-report baseline using original synthetic temperature readings and a fictional review protocol. No physical diagnosis, equipment command or live LLM inference is performed.

Read the definitions below. Identify the data boundary, the allowed operations and a condition that should stop execution. The executable reference is deliberately small enough to inspect.

In [ ]:
import math, statistics, hashlib, json
VERSION='thermal-workflow-1.0'
PROTOCOL={'id':'LAB-P1:v1','threshold_C':80.0,'comparison':'>=','scope':'Fictional original teaching protocol; no hardware authority'}
def stable_hash(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False,separators=(',',':')).encode()).hexdigest()
def analyze(request,protocol=PROTOCOL,bug=False):
    trace=[]
    def finish(status,reason=None,**kw):
        return dict(status=status,reason=reason,trace=trace,**kw)
    if type(request) is not dict or set(request)!={'case_id','unit','values'}:
        return finish('rejected','exact request fields required')
    if type(request['case_id']) is not str or not request['case_id'].strip():
        return finish('rejected','nonblank string case ID required')
    values=request['values']
    if type(request['unit']) is not str or request['unit'] not in ('C','F'):
        return finish('rejected','unit must be C or F')
    if type(values) is not list or not 2<=len(values)<=100 or not all(type(v) in (int,float) and abs(v)<=1e6 and math.isfinite(v) for v in values):
        return finish('rejected','2..100 finite bounded numeric readings required')
    trace.append({'stage':'validate','status':'ok','input_sha256':stable_hash(request)})
    c=[float(v) if request['unit']=='C' else (float(v)-32)*5/9 for v in values]
    trace.append({'stage':'convert','unit':'C','n':len(c)})
    mean=(sum(c)/(len(c)+1)) if bug else statistics.mean(c)
    sd=statistics.stdev(c)
    trace.append({'stage':'summarize','mean_C':mean,'sample_sd_C':sd})
    if (type(protocol) is not dict or set(protocol)!={'id','threshold_C','comparison','scope'} or
        type(protocol.get('id')) is not str or not protocol['id'].strip() or
        type(protocol.get('threshold_C')) not in (int,float) or abs(protocol['threshold_C'])>1e6 or not math.isfinite(protocol['threshold_C']) or
        protocol.get('comparison')!='>=' or type(protocol.get('scope')) is not str):
        return finish('abstain','approved class protocol unavailable',case_id=request['case_id'],mean_C=mean,sample_sd_C=sd)
    trace.append({'stage':'protocol','source_id':protocol['id'],'threshold_C':protocol['threshold_C']})
    decision_value=round(mean,10)
    status='review_required' if decision_value>=protocol['threshold_C'] else 'reported'
    trace.append({'stage':'report','status':status})
    return finish(status,case_id=request['case_id'],mean_C=mean,sample_sd_C=sd,decision_value_C=decision_value,
                  input_sha256=stable_hash(request),protocol_id=protocol['id'],version=VERSION,
                  limitation='Synthetic class readings and fictional protocol; not a physical safety conclusion.')

REQUEST={'case_id':'synthetic-A','unit':'C','values':[78.,80.,82.]}
print('Protocol:',PROTOCOL)

## Task 1. Predict the four contract outcomes

Classify low, equality, high and malformed requests before running. Reported does not mean physically safe.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** Equality triggers the inclusive >= rule. A malformed request has no claimed numeric result.

In [ ]:
responses['task1_prediction']=''  # Write before running the next cell.
responses['task1_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
requests=[{'case_id':'low','unit':'C','values':[70,72,74]}, {'case_id':'equal','unit':'C','values':[78,80,82]},
          {'case_id':'high','unit':'C','values':[81,82,83]}, {'case_id':'bad','values':[1,2]}]
results=[analyze(r) for r in requests]
assert [r['status'] for r in results]==['reported','review_required','review_required','rejected']
observations['task1']={'requests':requests,'results':results}
plot('case-means.png',['low','equal','high'],[r['mean_C'] for r in results[:3]],'Mean (C)','Synthetic repeated-reading cases; fictional threshold 80 C')
print([r['status'] for r in results])

## Task 2. Inspect provenance before prose

Locate case ID, input hash, protocol ID and implementation version, then explain the question each answers.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** A matching input digest establishes identity relative to canonical request bytes, not measurement authenticity.

In [ ]:
responses['task2_prediction']=''  # Write before running the next cell.
responses['task2_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
result=analyze(REQUEST)
observations['task2']={k:result[k] for k in ['case_id','input_sha256','protocol_id','version','limitation']}
assert result['input_sha256']==stable_hash(REQUEST) and result['protocol_id']=='LAB-P1:v1'
print(observations['task2'])

## Task 3. Check equivalent Celsius and Fahrenheit requests

Predict equal means and sample SDs. Keep distinct request hashes because raw input representations differ.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** The declared rounded comparison prevents a binary-rounding equality flip. It is a computational convention, not a measurement uncertainty model.

In [ ]:
responses['task3_prediction']=''  # Write before running the next cell.
responses['task3_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
f={'case_id':'synthetic-A-F','unit':'F','values':[172.4,176.,179.6]}
c_result=analyze(REQUEST); f_result=analyze(f)
assert math.isclose(c_result['mean_C'],f_result['mean_C'],abs_tol=1e-10)
assert math.isclose(c_result['sample_sd_C'],f_result['sample_sd_C'],abs_tol=1e-10)
assert c_result['status']==f_result['status']=='review_required'
observations['task3']={'C':c_result,'F':f_result,'mean_difference':f_result['mean_C']-c_result['mean_C']}
print(observations['task3'])

## Task 4. Reconstruct the sample spread

Show deviations, squared deviations and the n-1 denominator for the synthetic reference.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** This spread does not estimate systematic calibration bias or establish independence of repeated readings.

In [ ]:
responses['task4_prediction']=''  # Write before running the next cell.
responses['task4_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
a=np.array(REQUEST['values']); deviations=a-a.mean(); squares=deviations**2
manual=math.sqrt(float(squares.sum())/(len(a)-1))
assert manual==2 and math.isclose(analyze(REQUEST)['sample_sd_C'],manual)
observations['task4']={'deviations':deviations.tolist(),'squares':squares.tolist(),'denominator':len(a)-1,'sample_sd_C':manual}
plot('deviations.png',['r1','r2','r3'],deviations,'Deviation from mean (C)','Synthetic repeated readings, not independent deployment cases')
print(observations['task4'])

## Task 5. Stop at the invalid unit boundary

Change only the unit to an unsupported value and inspect which downstream stages are absent.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** The workflow must not convert, summarize or conclude before validating the input contract.

In [ ]:
responses['task5_prediction']=''  # Write before running the next cell.
responses['task5_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
bad=copy.deepcopy(REQUEST); bad['unit']='unknown'
failed=analyze(bad)
assert failed['status']=='rejected' and failed['trace']==[] and 'mean_C' not in failed
observations['task5']={'request':bad,'result':failed,'downstream_stages_present':[]}; print(observations['task5'])

## Task 6. Abstain without approved source evidence

Keep readings valid but remove the protocol. A known mean does not authorize a threshold-based conclusion.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** This is evidence unavailability, not malformed numeric input. No default threshold is silently substituted.

In [ ]:
responses['task6_prediction']=''  # Write before running the next cell.
responses['task6_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
missing=analyze(REQUEST,protocol=None)
assert missing['status']=='abstain' and missing['mean_C']==80 and 'protocol_id' not in missing
assert [t['stage'] for t in missing['trace']]==['validate','convert','summarize']
observations['task6']={'result':missing}; print(missing)

## Task 7. Expose a plausible denominator patch

The supplied buggy branch divides by n+1. Predict why it misses the equality review case before comparing.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** This is an intentionally supplied implementation defect, not an observed mistake from a real language model.

In [ ]:
responses['task7_prediction']=''  # Write before running the next cell.
responses['task7_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
fixture={'case_id':'boundary','unit':'C','values':[79,80,81]}
reference=analyze(fixture); buggy=analyze(fixture,bug=True)
assert reference['mean_C']==80 and buggy['mean_C']==60
assert reference['status']=='review_required' and buggy['status']=='reported'
observations['task7']={'fixture':fixture,'reference':reference,'buggy':buggy}; print(observations['task7'])

## Task 8. Save the input, protocol, source and result

Retain the unchanged reference alongside any experimental patch. Preserve actual source in the export.

1. Predict the outcome in the response cell below before running the analysis.
2. Run the next analysis cell. Read all printed fields, not just the last number.
3. Explain the mechanism using an observed value, case ID, rejection or source.
4. If an assertion fails unexpectedly, preserve the error and inspect the relevant contract; do not delete the assertion.

**Self-check:** Download the final ZIP before the runtime expires. Preserve source, not only a screenshot.

In [ ]:
responses['task8_prediction']=''  # Write before running the next cell.
responses['task8_explanation']=''  # Edit after inspecting actual evidence.

In [ ]:
save_json('request.json',REQUEST); save_json('protocol.json',PROTOCOL)
observations['task8']={'request_sha256':stable_hash(REQUEST),'protocol_sha256':stable_hash(PROTOCOL),
 'trace':analyze(REQUEST)['trace'],'scope':'Local deterministic baseline; original synthetic readings; fictional protocol.'}
assert set(observations)=={f'task{i}' for i in range(1,9)}
print(observations['task8'])

## Optional challenges

A: explicitly adapt Kelvin to the original C/F interface and retain regression checks. B: reverse repeated readings and inspect summary invariance versus input identity.

Predict first. Set `RUN_CHALLENGES=True` only if core tasks are complete. Record changes separately; these do not change the core completion count.

In [ ]:
RUN_CHALLENGES=False

optional={}
if RUN_CHALLENGES:
    # Explicit adapter: Kelvin is not silently accepted by the original C/F contract.
    kelvin={'case_id':'K-adapted','unit':'K','values':[351.15,353.15,355.15]}
    adapted={'case_id':kelvin['case_id'],'unit':'C','values':[v-273.15 for v in kelvin['values']]}
    assert analyze(kelvin)['status']=='rejected'
    kr=analyze(adapted); assert math.isclose(kr['mean_C'],80,abs_tol=1e-10) and kr['status']=='review_required'
    optional['A_kelvin_adapter']={'original':kelvin,'adapted':adapted,'result':kr}
    reversed_request=copy.deepcopy(REQUEST); reversed_request['values'].reverse()
    rr=analyze(reversed_request); original=analyze(REQUEST)
    assert math.isclose(rr['mean_C'],original['mean_C'],abs_tol=1e-10) and math.isclose(rr['sample_sd_C'],original['sample_sd_C'],abs_tol=1e-10)
    optional['B_order_invariance']={'result':rr,'note':'Numerical summary invariant; raw input digest and order differ.'}
print(optional)

## Final reflection and export

Fill the three reflections below. Keep one limitation even if every assertion passes. The export creates a draft while any of the 19 response fields are blank; ready for peer discussion checks completeness, not correctness. Download the ZIP via the link or the Colab Files sidebar before leaving the temporary runtime.

In [ ]:
responses['controlled_change']=''
responses['limitation']=''
responses['ai_disclosure']=''  # Write none if no AI assistance was used.

In [ ]:
(ARTIFACTS/'supplied-analysis-source.py').write_text(ANALYSIS_SOURCE)

# Edit response assignments above, then rerun this export cell only.
missing_responses=[k for k,v in responses.items() if type(v) is not str or not v.strip()]
record={'week':WEEK,'environment':ENV,'seed':SEED,'observations':observations,'optional':optional,
        'responses':responses,'missing_responses':missing_responses,
        'status':'draft' if missing_responses else 'ready for peer discussion',
        'completion_is_not_correctness':True,'policy':'In-class practice; no new weekly graded homework.'}
save_json('evidence.json',record)
# Snapshot source is supplied by the notebook generator, never invented from output.
(ARTIFACTS/'workflow-source.py').write_text(WORKFLOW_TEXT if WORKFLOW_TEXT else 'No shared thermal workflow in this unit. Inspect the complete notebook source.\n')
payloads=[p for p in ARTIFACTS.rglob('*') if p.is_file() and p not in [ARTIFACTS/'manifest.json',ARTIFACTS/'exercise-evidence.zip'] and not p.name.endswith('.ipynb')]
manifest={str(p.relative_to(ARTIFACTS)):hashlib.sha256(p.read_bytes()).hexdigest() for p in payloads}
save_json('manifest.json',manifest)
archive=ARTIFACTS/'exercise-evidence.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in payloads+[ARTIFACTS/'manifest.json']: z.write(p,arcname=str(p.relative_to(ARTIFACTS)))
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('Status:',record['status'],'; missing fields:',missing_responses)
print('Saved:',archive.resolve())
try:
    from IPython.display import display,FileLink
    display(FileLink(str(archive)))
except ImportError:
    pass
# Colab: open the Files sidebar, locate artifacts/weekXX/exercise-evidence.zip,
# use its menu to Download. Download before leaving the temporary runtime.

## Primary references and source boundaries

- [scikit-learn common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)
- [NIST Generative AI Profile](https://doi.org/10.6028/NIST.AI.600-1)

Original teaching fixtures, source passages and numerical protocol limits were authored for this class. They are not real hardware evidence. For a project extension, preserve your own edited notebook, code, data, versions and evaluation; do not relabel scaffold results as your own measured experiment.